# Documentation des Sources de Données (Projet AgriData Senegal)

Ce notebook explique en détail le rôle et le fonctionnement des trois scripts responsables de la génération et de la collecte de données, qui alimentent l'architecture Big Data via Apache Kafka :

1. `capteurs_sol.py`
2. `meteo_agri.py`
3. `production.py`

## 1. `capteurs_sol.py` : Simulation des Données Pédologiques (Sols)

Ce script simule les relevés de capteurs IoT virtuels plantés dans les sols agricoles de 14 régions du Sénégal.

### Fonctionnement :
- **Fréquence** : Envoi de données en temps réel. Une boucle infinie envoie de nouveaux payloads pour chaque région toutes les **3 secondes**.
- **Topic Kafka** : Pousse les données dans le topic Kafka `sol-capteurs`.
- **Mécanisme de Génération** : Il part de valeurs de base stables initialisées par région (par ex: *T=28°C, h=55%, pH=6.5* pour Kaolack). La fonction de mathématique `variation()` leur applique un cycle journalier régulier (avec une fonction `math.sin` étalée sur 24 heures) tout en lui additionnant un "bruit" statistique (avec `random.gauss`) pour imiter l'imprévisibilité de la nature.
- **Seuils d'Alertes** : Une variable `alerte=True` est intégrée dans le JSON si :
  - L'humidité passe sous les **20%** (Stress hydrique sévère).
  - Le pH passe **en dessous de 5.0 ou au-dessus de 8.5** (Acidité/Basicité critique).

In [ ]:
# Illustration du modèle mathématique de variation temporelle des capteurs
import random
import math

def simuler_variation(valeur_base, amplitude, temps_secondes):
    """Simule un cycle environnemental diurne-nocturne (sur 86400s) + bruit statique"""
    cycle = amplitude * math.sin(2 * math.pi * temps_secondes / 86400)
    bruit = random.gauss(0, amplitude * 0.1)
    return round(valeur_base + cycle + bruit, 2)

print("Temps t=1000s, T°C simulée pour une base de 28°C :", simuler_variation(28, 4, 1000), "°C")
print("Temps t=1000s, Humidité simulée pour 55% base :", simuler_variation(55, 8, 1000), "%")

## 2. `meteo_agri.py` : Collecte de Données Météorologiques

Ce script extrait des données climatiques externes relatives aux températures atmosphériques et pluviométrique des régions du projet.

### Fonctionnement :
- **Fréquence** : Envoi programmé en batch régulier, toutes les **60 secondes**.
- **Topic Kafka** : Pousse les données dans `meteo-agricole`.
- **Mécanisme d'Extraction** : Il utilise l'API tierce **OpenWeatherMap**.
  - Si la variable d'environnement `OWM_API_KEY` est fonctionnelle, le script envoie une requête API (`requests.get`) vers `api.openweathermap.org` pour chercher la météo exacte grâce aux coordonnées GPS (latitude/longitude) spécifiées pour chaque région.
  - Si aucune clé API n'est donnée (Fallback de secours), il génère automatiquement des approximations simulées en fonction de l'heure du système local (`time.time()`).
- **Seuils d'Alertes** : Le script identifie un état critique et émet une alerte dès lors que les précipitations enregistrées sur la dernière heure sont inférieures à **1 mm** ALORS QUE la température est supérieure à **35°C** (Condition conjuguée de sécheresse évidente).

## 3. `production.py` : Estimation des Rendements Agricoles

Ce script croise des données historiques réelles avec une simulation démographique de prévision des récoltes.

### Fonctionnement :
- **Fréquence** : Envoi régulier, toutes les **10 secondes**.
- **Topic Kafka** : Construit des rapports pour `production-agricole`.
- **Mécanisme Hybride (Historique + Simulé)** :
  1. Extrait la surface terrienne moyenne allouée pour la culture cible grâce au chargement du fichier de base de données **FAO** (existant en brut : `../data/fao_senegal.csv`).
  2. Étant donné que le CSV est au niveau national, la surface (en hectare) est re-scindée régionalement (divisée en 14 avec un coefficient modérateur aléatoire entre `0.5` et `1.5` pour varier l'échelle selon les régions).
  3. Pour le rendement par hectare, le modèle utilise une table prédéfinie (les rendements standard acceptables pour l'arachide, le coton, le riz) sur laquelle il applique une modification via distribution de Gauss (pour imiter le gain ou perte inopinée de récoltes).
  4. Le Produit Brut (`production_t`) est ainsi dérivé = Surface (`ha`) × Rendement (`T/ha`).
- **Seuils d'Alertes** : Une "alerte rendement" est déclenchée si le rendement final simulé est extraordinairement bas c'est-à-dire **sous 0.3 Tonnes / Hectare**. Cela signale statistiquement un risque de famine ou mal-production majeur.

In [1]:
# Aperçu stylisé de la modélisation des récoltes intégrée dans production.py
surface_nationale_fao = 520000  # hectares totaux exemple pour Arachide
rendement_moyen_t = 1.2         # T/ha basique 

def rendement_simule(base_agricole):
    variation = random.gauss(0, base_agricole * 0.1)
    return round(max(0.1, base_agricole + variation), 3)

# Pour une région (ex: Diourbel) 
surface_simule_region = round((surface_nationale_fao / 14) * random.uniform(0.5, 1.5), 0)
rendement_local = rendement_simule(rendement_moyen_t)
production_totale = round(surface_simule_region * rendement_local, 1)

alerte = rendement_local < 0.3

print("Exemple d'agrégat pour la région locale:")
print(f"- Surface allouée: {surface_simule_region} ha")
print(f"- Rendement final: {rendement_local} T/ha {'(ALERTE)' if alerte else ''}")
print(f"- Production estimée dans Kafka: {production_totale} Tonnes")

NameError: name 'random' is not defined